# Tutorial 2: Differential Privacy with `clipped_grad`

**Goal**: Learn what differential privacy (DP) is, understand the DP-SGD algorithm, and use Opaque's `clipped_grad()` function to implement it.

**Prerequisites**: 
- [Tutorial 1: Understanding Per-Sample Gradients](01_understanding_per_sample_gradients.ipynb)
- Basic understanding of privacy concepts

**What you'll learn**:
1. What is differential privacy and why does it matter?
2. The DP-SGD algorithm step-by-step
3. What is gradient clipping and why is it necessary?
4. Using Opaque's `clipped_grad()` for efficient DP training
5. Understanding sensitivity and privacy guarantees
6. Practical examples with real training

---

## Part 1: What is Differential Privacy?

### The Privacy Problem

When you train a machine learning model on sensitive data (medical records, private messages, financial data), the model might **memorize** specific training examples.

**Example Attack**: An adversary could:
1. Train a model on dataset D
2. Add one specific person's record
3. Train again on D ∪ {record}
4. Compare the models to infer information about that person!

### Differential Privacy: The Solution

**Formal Definition** (simplified):

An algorithm is (ε, δ)-differentially private if for any two datasets D₁ and D₂ that differ in **one single record**:

```
P[Algorithm(D₁) ∈ S] ≤ e^ε × P[Algorithm(D₂) ∈ S] + δ
```

**Intuitive meaning**:
- An adversary **can't tell** whether your data was in the training set or not
- **ε (epsilon)**: Privacy budget (smaller = more private)
  - Typical values: 1.0 (strong) to 10.0 (weak)
- **δ (delta)**: Probability of privacy failure
  - Typical values: 10⁻⁵ to 10⁻⁶

### How Do We Achieve This?

**Two key mechanisms**:
1. **Clip gradients**: Bound each example's influence on the model
2. **Add noise**: Randomize the updates to hide individual contributions

Let's see how this works!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from opaque.clipping import clipped_grad
from opaque.pytree_utils import global_norm

torch.manual_seed(42)
np.random.seed(42)

## Part 2: The Problem - Unbounded Gradient Norms

Let's create a dataset where one example is an **outlier**. This outlier will have a very large gradient.

In [ ]:
# Create dataset with an outlier
X_train = torch.tensor([
    [1.0], [2.0], [3.0], [4.0], [5.0],  # Normal points
    [6.0], [7.0], [8.0], [9.0], 
    [100.0]  # OUTLIER!
])

# Linear relationship: y = 2x + 1, but outlier has huge y
y_train = 2 * X_train + 1
y_train[-1] = 1000.0  # Outlier label

print("Training data:")
print("X:", X_train.squeeze().tolist())
print("y:", y_train.squeeze().tolist())

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X_train[:-1].numpy(), y_train[:-1].numpy(), 
            s=100, alpha=0.7, label='Normal points')
plt.scatter(X_train[-1].numpy(), y_train[-1].numpy(), 
            s=200, color='red', marker='*', label='OUTLIER')
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Dataset with Outlier', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("\n⚠️ The outlier will dominate the gradients!")

### Computing Per-Sample Gradients (Review)

Let's compute per-sample gradients and see how large the outlier's gradient is:

In [ ]:
from torch.func import grad, vmap

# Simple linear model parameters
params = {
    'weight': torch.randn(1, 1, requires_grad=True),
    'bias': torch.randn(1, requires_grad=True)
}

def loss_fn(params, x, y):
    """Loss for a single example."""
    pred = x @ params['weight'].T + params['bias']
    return F.mse_loss(pred, y)

# Create per-sample gradient function
grad_fn = grad(loss_fn)
per_sample_grad_fn = vmap(grad_fn, in_dims=(None, 0, 0))

# Compute per-sample gradients
grads = per_sample_grad_fn(params, X_train, y_train)

# Compute gradient norms
grad_norms = []
for i in range(len(X_train)):
    grad_i = {
        'weight': grads['weight'][i],
        'bias': grads['bias'][i]
    }
    norm = global_norm(grad_i).item()
    grad_norms.append(norm)

# Visualize gradient norms
plt.figure(figsize=(12, 5))
colors = ['blue'] * 9 + ['red']  # Red for outlier
bars = plt.bar(range(10), grad_norms, color=colors, alpha=0.7)
plt.xlabel('Example Index', fontsize=12)
plt.ylabel('Gradient Norm (L2)', fontsize=12)
plt.title('Per-Sample Gradient Norms', fontsize=14)
plt.axhline(y=np.mean(grad_norms[:-1]), color='blue', linestyle='--', 
            label=f'Avg (normal): {np.mean(grad_norms[:-1]):.2f}')
plt.axhline(y=grad_norms[-1], color='red', linestyle='--', 
            label=f'Outlier: {grad_norms[-1]:.2f}')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.show()

print(f"\nNormal examples gradient norm: ~{np.mean(grad_norms[:-1]):.2f}")
print(f"Outlier gradient norm: {grad_norms[-1]:.2f}")
print(f"\n🚨 The outlier's gradient is {grad_norms[-1]/np.mean(grad_norms[:-1]):.0f}x larger!")
print("   This would completely dominate model updates!")

## Part 3: Gradient Clipping - Bounding the Influence

### The Key Idea

**Gradient clipping** ensures no single example can have too much influence:

```
if ||gradient|| > max_norm:
    gradient = gradient * (max_norm / ||gradient||)
```

This **rescales** the gradient to have norm exactly `max_norm`.

### Why This Helps Privacy

1. **Bounded sensitivity**: Each example can change the sum by at most `max_norm`
2. **Outliers can't dominate**: Their influence is capped
3. **Enables noise**: We know how much noise to add (proportional to max_norm)

Let's implement this manually first:

In [ ]:
def clip_gradient(grad_dict, max_norm):
    """
    Clip gradient to maximum L2 norm.
    
    Args:
        grad_dict: Dict of gradients {name: tensor}
        max_norm: Maximum allowed L2 norm
    
    Returns:
        Clipped gradient dict, actual norm
    """
    # Compute L2 norm across all parameters
    norm = global_norm(grad_dict)
    
    # Clip if necessary
    if norm > max_norm:
        scale = max_norm / norm
        clipped = {k: v * scale for k, v in grad_dict.items()}
        return clipped, norm.item()
    else:
        return grad_dict, norm.item()

# Clip all gradients to max_norm = 10.0
max_norm = 10.0
clipped_grads = []
original_norms = []
clipped_norms = []

for i in range(len(X_train)):
    grad_i = {
        'weight': grads['weight'][i],
        'bias': grads['bias'][i]
    }
    clipped, orig_norm = clip_gradient(grad_i, max_norm)
    clipped_grads.append(clipped)
    original_norms.append(orig_norm)
    clipped_norms.append(global_norm(clipped).item())

# Visualize before/after clipping
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Before clipping
ax1.bar(range(10), original_norms, color=colors, alpha=0.7)
ax1.axhline(y=max_norm, color='green', linestyle='--', linewidth=2, label=f'Clip threshold: {max_norm}')
ax1.set_xlabel('Example Index', fontsize=11)
ax1.set_ylabel('Gradient Norm', fontsize=11)
ax1.set_title('BEFORE Clipping', fontsize=13)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# After clipping
ax2.bar(range(10), clipped_norms, color=colors, alpha=0.7)
ax2.axhline(y=max_norm, color='green', linestyle='--', linewidth=2, label=f'Clip threshold: {max_norm}')
ax2.set_xlabel('Example Index', fontsize=11)
ax2.set_ylabel('Gradient Norm', fontsize=11)
ax2.set_title('AFTER Clipping', fontsize=13)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(ax1.get_ylim())  # Same y-axis for comparison

plt.tight_layout()
plt.show()

print(f"Outlier gradient norm: {original_norms[-1]:.2f} → {clipped_norms[-1]:.2f}")
print(f"\n✂️ The outlier's gradient was clipped from {original_norms[-1]:.0f} to {max_norm}!")
print(f"   Now its influence is bounded.")

## Part 4: Introducing `clipped_grad()` - The Easy Way

Manual clipping is tedious! Opaque's `clipped_grad()` does everything for you:

1. ✅ Computes per-sample gradients (using vmap)
2. ✅ Clips each gradient to max norm
3. ✅ Sums the clipped gradients
4. ✅ Optionally returns per-sample norms and values

### Basic Usage

In [ ]:
# Create clipped gradient function
clipped_grad_fn = clipped_grad(
    loss_fn,              # Loss function for single example
    argnums=0,            # Compute gradients w.r.t. first arg (params)
    batch_argnums=1,      # Second arg (x) has batch dimension
    l2_clip_norm=10.0,    # Maximum gradient norm per example
    normalize_by=len(X_train),  # Divide by batch size (for averaging)
)

# Compute clipped gradients for entire batch!
clipped_batch_grad = clipped_grad_fn(params, X_train, y_train)

print("Clipped batch gradient (summed and averaged):")
print(f"  weight.grad: {clipped_batch_grad['weight']}")
print(f"  bias.grad: {clipped_batch_grad['bias']}")
print(f"\n✨ One function call computed per-sample grads, clipped them, and summed!")

### Accessing Auxiliary Information

We can also get per-sample gradient norms and loss values:

In [ ]:
# Create clipped gradient function with auxiliary outputs
clipped_grad_fn_with_info = clipped_grad(
    loss_fn,
    argnums=0,
    batch_argnums=1,
    l2_clip_norm=10.0,
    normalize_by=len(X_train),
    return_grad_norms=True,  # Return per-sample gradient norms
    return_values=True,       # Return per-sample loss values
)

# Compute with auxiliary info
clipped_batch_grad, aux_output = clipped_grad_fn_with_info(params, X_train, y_train)

print("Per-sample gradient norms (BEFORE clipping):")
print(aux_output.grad_norms)
print(f"\nPer-sample loss values:")
print(aux_output.values)

# Visualize which examples were clipped
was_clipped = aux_output.grad_norms > 10.0
print(f"\n✂️ Examples that were clipped: {was_clipped.nonzero().squeeze().tolist()}")
print(f"   (Index {len(X_train)-1} is our outlier)")

## Part 5: The Complete DP-SGD Algorithm

Now let's put it all together with noise! Here's the full DP-SGD algorithm:

```
1. Compute per-sample gradients
2. Clip each gradient to max_norm
3. Sum clipped gradients
4. Add Gaussian noise: noise ~ N(0, σ² * max_norm²)
5. Update parameters
```

**Note**: Noise injection (Stage 2) is coming soon in Opaque! For now, we'll add noise manually.

In [ ]:
def add_gaussian_noise(grad_dict, noise_multiplier, l2_clip_norm):
    """
    Add Gaussian noise to gradients for DP.
    
    Args:
        grad_dict: Gradient dictionary
        noise_multiplier: Noise scale (σ in DP theory)
        l2_clip_norm: Clipping norm (C in DP theory)
    
    Returns:
        Noisy gradient dict
    """
    noisy_grad = {}
    for k, v in grad_dict.items():
        noise_scale = noise_multiplier * l2_clip_norm
        noise = torch.randn_like(v) * noise_scale
        noisy_grad[k] = v + noise
    return noisy_grad

# Hyperparameters
max_norm = 1.0          # Clipping threshold
noise_multiplier = 1.1  # Noise scale (higher = more privacy, less accuracy)
learning_rate = 0.01
batch_size = len(X_train)

# Create DP gradient function
dp_grad_fn = clipped_grad(
    loss_fn,
    argnums=0,
    batch_argnums=1,
    l2_clip_norm=max_norm,
    normalize_by=batch_size,
    return_grad_norms=True,
)

# Single DP-SGD step
clipped_grad_batch, aux = dp_grad_fn(params, X_train, y_train)
print("Step 1-3: Clipped batch gradient:")
print(f"  weight: {clipped_grad_batch['weight'].item():.4f}")
print(f"  bias: {clipped_grad_batch['bias'].item():.4f}")

# Add noise (Step 4)
noisy_grad = add_gaussian_noise(clipped_grad_batch, noise_multiplier, max_norm)
print("\nStep 4: After adding noise:")
print(f"  weight: {noisy_grad['weight'].item():.4f}")
print(f"  bias: {noisy_grad['bias'].item():.4f}")

# Update parameters (Step 5)
params_updated = {
    k: v - learning_rate * noisy_grad[k] 
    for k, v in params.items()
}
print("\nStep 5: Updated parameters:")
print(f"  weight: {params['weight'].item():.4f} → {params_updated['weight'].item():.4f}")
print(f"  bias: {params['bias'].item():.4f} → {params_updated['bias'].item():.4f}")

print("\n🔒 This is differentially private training!")
print(f"   Privacy guarantee: ({noise_multiplier}, δ)-DP per step")
print(f"   (Exact ε depends on number of steps and is computed via privacy accounting)")

## Part 6: Training a Model with DP-SGD

Let's train a simple model using DP-SGD and compare it to standard SGD:

In [ ]:
# Create training data (without the extreme outlier)
torch.manual_seed(42)
X = torch.linspace(0, 10, 100).unsqueeze(1)
y = 2 * X + 1 + torch.randn_like(X) * 0.5  # y = 2x + 1 + noise

# Split into train/test
split = 80
X_train_data, X_test_data = X[:split], X[split:]
y_train_data, y_test_data = y[:split], y[split:]

print(f"Training samples: {len(X_train_data)}")
print(f"Test samples: {len(X_test_data)}")

In [ ]:
def train_model(X, y, use_dp=False, num_epochs=50, lr=0.01, max_norm=1.0, noise_mult=1.1):
    """
    Train linear model with standard SGD or DP-SGD.
    
    Returns:
        Trained parameters, loss history
    """
    # Initialize parameters
    params = {
        'weight': torch.randn(1, 1) * 0.1,
        'bias': torch.randn(1) * 0.1
    }
    
    loss_history = []
    
    if use_dp:
        # DP-SGD: Use clipped_grad
        grad_fn = clipped_grad(
            loss_fn,
            argnums=0,
            batch_argnums=1,
            l2_clip_norm=max_norm,
            normalize_by=len(X),
        )
    else:
        # Standard SGD: Average gradients
        from torch.func import grad, vmap
        grad_single = grad(loss_fn)
        grad_fn_batch = vmap(grad_single, in_dims=(None, 0, 0))
    
    for epoch in range(num_epochs):
        if use_dp:
            # DP-SGD step
            grads = grad_fn(params, X, y)
            # Add noise
            grads = add_gaussian_noise(grads, noise_mult, max_norm)
        else:
            # Standard SGD step
            per_sample_grads = grad_fn_batch(params, X, y)
            # Average gradients (no clipping, no noise)
            grads = {
                k: v.mean(dim=0) for k, v in per_sample_grads.items()
            }
        
        # Update parameters
        params = {
            k: v - lr * grads[k] for k, v in params.items()
        }
        
        # Compute loss
        with torch.no_grad():
            preds = X @ params['weight'].T + params['bias']
            loss = F.mse_loss(preds, y)
            loss_history.append(loss.item())
    
    return params, loss_history

# Train both models
print("Training Standard SGD...")
params_std, loss_std = train_model(X_train_data, y_train_data, use_dp=False)

print("Training DP-SGD...")
params_dp, loss_dp = train_model(X_train_data, y_train_data, use_dp=True, 
                                   max_norm=1.0, noise_mult=1.1)

print("\n✅ Training complete!")

In [ ]:
# Visualize training curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(loss_std, label='Standard SGD', linewidth=2)
plt.plot(loss_dp, label='DP-SGD (ε≈?)', linewidth=2, alpha=0.8)
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Training Loss', fontsize=11)
plt.title('Training Loss Curves', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# Predictions
with torch.no_grad():
    pred_std = X_test_data @ params_std['weight'].T + params_std['bias']
    pred_dp = X_test_data @ params_dp['weight'].T + params_dp['bias']

plt.scatter(X_test_data.numpy(), y_test_data.numpy(), alpha=0.5, label='True data', s=50)
plt.plot(X_test_data.numpy(), pred_std.numpy(), 'g-', linewidth=2, label='Standard SGD')
plt.plot(X_test_data.numpy(), pred_dp.numpy(), 'r--', linewidth=2, label='DP-SGD')
plt.xlabel('X', fontsize=11)
plt.ylabel('y', fontsize=11)
plt.title('Test Set Predictions', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print learned parameters
print("Learned parameters:")
print(f"\nTrue model: y = 2.0x + 1.0")
print(f"\nStandard SGD: y = {params_std['weight'].item():.3f}x + {params_std['bias'].item():.3f}")
print(f"DP-SGD:       y = {params_dp['weight'].item():.3f}x + {params_dp['bias'].item():.3f}")

print("\n💡 Observations:")
print("   1. DP-SGD converges slower (due to noise)")
print("   2. Final accuracy is similar for this simple problem")
print("   3. DP-SGD provides privacy guarantee - Standard SGD does not!")

## Part 7: Understanding the Privacy-Utility Tradeoff

Let's experiment with different privacy levels:

In [ ]:
# Train with different noise levels
noise_levels = [0.0, 0.5, 1.0, 2.0, 5.0]
results = []

for noise_mult in noise_levels:
    params, loss_hist = train_model(
        X_train_data, y_train_data, 
        use_dp=True, 
        noise_mult=noise_mult,
        num_epochs=100
    )
    results.append({
        'noise_mult': noise_mult,
        'final_loss': loss_hist[-1],
        'weight': params['weight'].item(),
        'bias': params['bias'].item()
    })

# Visualize tradeoff
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss vs noise
noise_vals = [r['noise_mult'] for r in results]
losses = [r['final_loss'] for r in results]
ax1.plot(noise_vals, losses, 'o-', linewidth=2, markersize=8)
ax1.set_xlabel('Noise Multiplier (σ)', fontsize=12)
ax1.set_ylabel('Final Training Loss', fontsize=12)
ax1.set_title('Privacy vs Utility Tradeoff', fontsize=14)
ax1.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='Typical: σ=1.0')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Parameter accuracy
weights = [r['weight'] for r in results]
ax2.plot(noise_vals, weights, 'o-', linewidth=2, markersize=8, label='Learned weight')
ax2.axhline(y=2.0, color='green', linestyle='--', linewidth=2, label='True weight: 2.0')
ax2.set_xlabel('Noise Multiplier (σ)', fontsize=12)
ax2.set_ylabel('Learned Weight', fontsize=12)
ax2.set_title('Parameter Accuracy vs Privacy', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Privacy-Utility Tradeoff:")
print("\nNoise σ  | Final Loss | Learned Weight | Privacy")
print("-" * 60)
for r in results:
    privacy = "None" if r['noise_mult'] == 0 else "✓ DP"
    print(f"{r['noise_mult']:5.1f}   | {r['final_loss']:10.4f} | {r['weight']:14.3f} | {privacy}")

print("\n💡 Key insight: More privacy (higher σ) → worse accuracy")
print("   Must choose σ based on privacy needs vs accuracy requirements")

## Part 8: Advanced Features of `clipped_grad`

### Feature 1: PyTree Parameters (Nested Dicts)

Real models have many parameters. `clipped_grad` works with **PyTrees** (nested dicts):

In [ ]:
# Multi-layer model parameters (PyTree structure)
params_mlp = {
    'layer1': {
        'weight': torch.randn(10, 1) * 0.1,
        'bias': torch.randn(10) * 0.1
    },
    'layer2': {
        'weight': torch.randn(1, 10) * 0.1,
        'bias': torch.randn(1) * 0.1
    }
}

def mlp_loss(params, x, y):
    """2-layer MLP loss."""
    h = F.relu(x @ params['layer1']['weight'].T + params['layer1']['bias'])
    pred = h @ params['layer2']['weight'].T + params['layer2']['bias']
    return F.mse_loss(pred, y)

# Compute clipped gradients for MLP
mlp_grad_fn = clipped_grad(
    mlp_loss,
    argnums=0,
    batch_argnums=1,
    l2_clip_norm=1.0,
    normalize_by=len(X_train_data),
)

grads_mlp = mlp_grad_fn(params_mlp, X_train_data, y_train_data)

print("Clipped gradients for MLP (PyTree):")
print(f"  layer1.weight shape: {grads_mlp['layer1']['weight'].shape}")
print(f"  layer1.bias shape: {grads_mlp['layer1']['bias'].shape}")
print(f"  layer2.weight shape: {grads_mlp['layer2']['weight'].shape}")
print(f"  layer2.bias shape: {grads_mlp['layer2']['bias'].shape}")
print("\n✨ Works seamlessly with nested parameter structures!")

### Feature 2: Sensitivity Tracking

`clipped_grad` returns a `BoundedSensitivityCallable` that tracks the L2 sensitivity:

In [ ]:
# Create clipped grad function
cg = clipped_grad(
    loss_fn,
    argnums=0,
    batch_argnums=1,
    l2_clip_norm=1.0,
    rescale_to_unit_norm=False,  # Sensitivity = l2_clip_norm
    normalize_by=10.0,
)

print(f"Type: {type(cg)}")
print(f"L2 sensitivity bound: {cg.l2_norm_bound}")
print(f"Has auxiliary output: {cg.has_aux}")

print("\n💡 Sensitivity tells us how much noise to add:")
print(f"   Noise scale = sensitivity × noise_multiplier")
print(f"   = {cg.l2_norm_bound} × 1.1 = {cg.l2_norm_bound * 1.1}")

### Feature 3: Pre-Clipping Transform (LoRA use case)

You can apply a transformation to gradients **before** clipping. Useful for LoRA!

In [ ]:
# Example: Only clip LoRA parameters
def select_lora_params(grad_dict):
    """Only keep gradients for LoRA adapters."""
    # In real LoRA, you'd filter for adapter weights only
    # Here we just demonstrate the concept
    return {
        k: v if 'weight' in k else torch.zeros_like(v)
        for k, v in grad_dict.items()
    }

cg_lora = clipped_grad(
    loss_fn,
    argnums=0,
    batch_argnums=1,
    l2_clip_norm=1.0,
    normalize_by=10.0,
    pre_clipping_transform=select_lora_params,  # Applied before clipping!
)

grads_lora = cg_lora(params, X_train[:10], y_train[:10])
print("Gradients with pre-clipping transform:")
print(f"  weight.grad: {grads_lora['weight'].item():.4f} (non-zero)")
print(f"  bias.grad: {grads_lora['bias'].item():.4f} (zeroed out)")
print("\n💡 Useful for fine-tuning only specific parameters (like LoRA adapters)!")

## Summary

### What We Learned

1. **Differential Privacy**: Mathematical guarantee that model doesn't memorize individuals
2. **DP-SGD Algorithm**:
   - Compute per-sample gradients
   - Clip each gradient to max norm
   - Add calibrated Gaussian noise
   - Update parameters

3. **Why Clipping?**: Bounds each example's influence → enables noise calibration

4. **Using `clipped_grad()`**:
   ```python
   cg = clipped_grad(
       loss_fn,
       argnums=0,
       batch_argnums=1,
       l2_clip_norm=1.0,
       normalize_by=batch_size,
   )
   grads = cg(params, X_batch, y_batch)
   ```

5. **Privacy-Utility Tradeoff**: More privacy (higher noise) → lower accuracy

### The Complete Picture

```
Standard SGD:           DP-SGD (Opaque):
┌─────────────────┐    ┌─────────────────┐
│ Compute batch   │    │ clipped_grad()  │
│ gradient        │    │  - Per-sample   │
│ (averaged)      │    │  - Clip each    │
└────────┬────────┘    │  - Sum clipped  │
         │             └────────┬────────┘
         │                      │
         v                      v
┌─────────────────┐    ┌─────────────────┐
│ Update params   │    │ Add noise       │
└─────────────────┘    └────────┬────────┘
                                 │
                                 v
                        ┌─────────────────┐
                        │ Update params   │
                        └─────────────────┘
                                 │
                        ✓ Privacy guarantee!
```

### Next Steps

1. **Privacy Accounting** (Stage 3): Compute exact (ε, δ) for your training
2. **Noise Module** (Stage 2): Automated noise injection in Opaque
3. **LoRA Integration** (Stage 4): Fine-tune LLMs with DP guarantees

### Resources

- [Opaque Documentation](https://opaque.readthedocs.io/)
- [DP-SGD Paper](https://arxiv.org/abs/1607.00133) (Abadi et al. 2016)
- [JAX-Privacy](https://github.com/google-deepmind/jax_privacy)
- [DP-Accounting Library](https://github.com/google/differential-privacy)

🎉 **Congratulations!** You now understand how to implement differentially private training with gradient clipping!